In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

from PIL import Image

from scipy.ndimage import uniform_filter

from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import cv2
import string
import os
from tqdm import tqdm

In [ ]:
# Config for Segmented Character
CHAR_IMAGE_HEIGHT = 64
CHAR_IMAGE_WIDTH = 64

In [ ]:
CHARSET = string.digits + string.ascii_lowercase

CHAR_TO_IDX = {'<UNK>': 0}

for char in CHARSET:
    if char not in CHAR_TO_IDX:
        CHAR_TO_IDX[char] = len(CHAR_TO_IDX)

IDX_TO_CHAR = {idx: char for char, idx in CHAR_TO_IDX.items()}

NUM_CLASSES = len(CHAR_TO_IDX)

print(f"Number of classes (vocabulary size): {NUM_CLASSES}")

Number of classes (vocabulary size): 37


In [ ]:
class CaptchaSegmenter:
    """Segments CAPTCHA images into individual character images"""

    def __init__(self, char_width=CHAR_IMAGE_WIDTH, char_height=CHAR_IMAGE_HEIGHT):
        self.char_width = char_width
        self.char_height = char_height

    def remove_lines(self, img) -> np.ndarray:
        img_grey = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        lines = np.where(img_grey == 0, 255, 0).astype(np.uint8)
        # Expand lines to 3 channels
        lines = cv2.merge([lines, lines, lines])
        processed = img + lines

        # Convert to float for computation
        img = img.astype(float)

        # Compute local mean for each channel using a 3x3 filter
        avg_img = np.zeros_like(img)
        for c in range(3):
            avg_img[..., c] = uniform_filter(processed[..., c], size=3, mode='reflect')

        # Create mask for black pixels
        mask_black = np.all(img == 0, axis=-1)

        # Replace black pixels with local averages
        img[mask_black] = avg_img[mask_black]

        # Clip and convert back
        return np.clip(img, 0, 255).astype(np.uint8)

    def segment_by_color(
        self,
        img: np.ndarray,
        eps: float = 0.5,           # DBSCAN neighborhood radius (tuned)
        min_pixel_area: int = 31,   # Minimum pixel count per final blob
        db_min_samples: int = 27,   # Minimum pixels to form a cluster
        spatial_weight: float = 0.1  # Controls importance of x,y distance
    ) -> tuple[int, list[np.ndarray]]:
        """
        Segments a CAPTCHA image using spatial + color (hue) clustering.

        - Converts image to HSV and filters out white background.
        - Maps hue to (cos, sin) to handle circular hue wraparound.
        - Clusters in 4D space: (x, y, cos(hue), sin(hue)) using DBSCAN.
        - Merges small gaps vertically to connect 'i' stems/dots.

        Args:
            img: Input RGB (or BGR) image as NumPy array.
            eps: DBSCAN neighborhood size (after scaling).
            min_pixel_area: Minimum size for a valid character blob.
            db_min_samples: Minimum samples for DBSCAN cluster.
            spatial_weight: Relative importance of spatial vs color distance.
        Returns:
            (character_count, list_of_character_images)
        """
        # --- 1. Convert to HSV ---
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        s_channel = hsv[:, :, 1]
        v_channel = hsv[:, :, 2]

        # --- 2. Filter out white background ---
        s_thresh = 5
        v_thresh = 250
        bg_mask = cv2.bitwise_and(
            cv2.compare(s_channel, s_thresh, cv2.CMP_LT),
            cv2.compare(v_channel, v_thresh, cv2.CMP_GT)
        )
        fg_mask = cv2.bitwise_not(bg_mask)

        y_coords, x_coords = np.where(fg_mask > 0)
        if len(y_coords) < min_pixel_area:
            return 0, []

        # --- 3. Get hue values and map to circular coords ---
        hues = hsv[y_coords, x_coords, 0].astype(np.float32) / 180.0  # normalize hue to [0,1]
        hue_x = np.cos(2 * np.pi * hues)
        hue_y = np.sin(2 * np.pi * hues)

        # --- 4. Combine features: spatial + color ---
        features = np.column_stack([
            y_coords,
            x_coords,
            hue_x,
            hue_y
        ])

        # --- 5. Normalize features (to make x,y comparable to hue) ---
        features_scaled = StandardScaler().fit_transform(features)
        features_scaled[:, :2] *= spatial_weight  # re-apply spatial weight after scaling

        # --- 6. DBSCAN clustering ---
        db = DBSCAN(eps=eps, min_samples=db_min_samples).fit(features_scaled)
        labels = db.labels_
        unique_labels = set(labels)

        char_images = []
        char_bboxes = []

        # --- 7. Build masks for each cluster ---
        for label in unique_labels:
            if label == -1:
                continue  # skip noise

            cluster_mask = np.zeros(img.shape[:2], dtype="uint8")
            cluster_idx = np.where(labels == label)[0]
            cluster_y = y_coords[cluster_idx]
            cluster_x = x_coords[cluster_idx]
            cluster_mask[cluster_y, cluster_x] = 255

            # --- 8. Morphological close (vertical kernel) ---
            kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 5))
            closed_mask = cv2.morphologyEx(cluster_mask, cv2.MORPH_CLOSE, kernel, iterations=1)

            # --- 9. Find connected components (characters) ---
            num_comp, comp_labels, stats, _ = cv2.connectedComponentsWithStats(closed_mask, 8, cv2.CV_32S)
            if num_comp <= 1:
                continue

            for i in range(1, num_comp):
                area = stats[i, cv2.CC_STAT_AREA]
                if area < min_pixel_area:
                    continue

                final_mask = (comp_labels == i)
                y_pix, x_pix = np.where(final_mask)
                if len(x_pix) == 0:
                    continue

                x_min = x_pix.min()

                # Create character image
                char_img = np.full_like(img, 255, dtype=np.uint8)
                char_img[final_mask] = img[final_mask]

                char_images.append(char_img)
                char_bboxes.append(x_min)

        # --- 10. Sort characters left-to-right ---
        if char_bboxes:
            sorted_idx = np.argsort(char_bboxes)
            char_images = [char_images[i] for i in sorted_idx]

        return len(char_images), char_images

    def segment_by_value(
        self,
        img: np.ndarray,
        eps: int = 5,  # <-- Tuned parameter
        min_pixel_area: int = 31,  # <-- Tuned parameter
        db_min_samples: int = 27, # <-- Tuned parameter
    ) -> tuple[int, list[np.ndarray]]:
        """
        Segments a CAPTCHA image using color clustering:
        - Uses a VERTICAL kernel (3,5) to connect 'i' dots/stems
        without merging horizontally-adjacent chars (like '7bm').

        Args:
            img: The input RGB image (as a NumPy array).
            eps: The max distance (eps) for one cluster.
            min_pixel_area: Min pixels for a *final spatial component*.
            db_min_samples: Min pixels to form a *color cluster*.
            spatial_weight: Weighting for spatial vs color distance.

        Returns:
            A tuple of (character_count, list_of_character_images).
        """

        # --- 1. Convert to HSV ---
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        s_channel = hsv[:, :, 1]
        v_channel = hsv[:, :, 2]

        # --- 2. Filter out white background ---
        s_thresh = 5
        v_thresh = 250
        bg_mask = cv2.bitwise_and(
            cv2.compare(s_channel, s_thresh, cv2.CMP_LT),
            cv2.compare(v_channel, v_thresh, cv2.CMP_GT)
        )
        fg_mask = cv2.bitwise_not(bg_mask)

        y_coords, x_coords = np.where(fg_mask > 0)
        if len(y_coords) < min_pixel_area:
            return 0, []

        values = hsv[y_coords, x_coords, 2].astype(np.float32)
        features = values.reshape(-1, 1)

        # --- 6. DBSCAN clustering ---
        db = DBSCAN(eps=eps, min_samples=db_min_samples).fit(features)
        labels = db.labels_
        unique_labels = set(labels)

        temp_char_images = []
        temp_char_bboxes = []
        char_images = []
        char_bboxes = []

        # 5. Create an image for each color cluster (character)
        for label in unique_labels:
            if label == -1: # Skip noise
                continue

            cluster_mask = np.zeros(img.shape[:2], dtype="uint8")

            cluster_indices = np.where(labels == label)[0]
            cluster_y = y_coords[cluster_indices]
            cluster_x = x_coords[cluster_indices]

            cluster_mask[cluster_y, cluster_x] = 255

            # 6. Clean up the mask (connect nearby parts of the same color)

            # A (5, 7) kernel is 5px wide and 7px tall.
            # It's more aggressive at closing VERTICAL gaps (like 'i')
            # than horizontal gaps (like '7bm').
            kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 7))
            closed_mask = cv2.morphologyEx(cluster_mask, cv2.MORPH_CLOSE, kernel, iterations=1)

            # 7. Find ALL characters from the color cluster mask
            num_components, comp_labels, stats, _ = \
                cv2.connectedComponentsWithStats(closed_mask, 8, cv2.CV_32S)

            if num_components <= 1: # Only background
                continue

            # Loop through all found components (skip 0, the background)
            for i in range(1, num_components):

                # 8. Filter small components by area
                area = stats[i, cv2.CC_STAT_AREA]
                if area < min_pixel_area:
                    continue

                # 9. Extract this component as a character
                final_mask = (comp_labels == i)

                _, x_pix = np.where(final_mask)
                if len(x_pix) == 0:
                    continue

                x_min = x_pix.min()

                char_img = np.full_like(img, 255, dtype=np.uint8)
                char_img[final_mask] = img[final_mask]

                temp_char_images.append(char_img)
                temp_char_bboxes.append(x_min)

        pixel_counts = [np.sum((img < 255).any(axis=-1)) for img in temp_char_images]

        if len(pixel_counts) == 0:
            return 0, []

        pixel_area_threshold = np.max(pixel_counts) * 0.7

        for img, bbox, pcount in zip(temp_char_images, temp_char_bboxes, pixel_counts):
            if pcount >= pixel_area_threshold:
                char_images.append(img)
                char_bboxes.append(bbox)

        # 10. Sort characters left-to-right
        if char_bboxes:
            sorted_indices = np.argsort(char_bboxes)
            char_images = [char_images[i] for i in sorted_indices]

        return len(char_images), char_images

    def detect_chars_and_sizes(
            self,
            img: np.ndarray,
            min_area: int = 20,
            connectivity: int = 8,
            s_thresh: int = 5,
            v_thresh: int = 250,
            display: bool = True
    ) -> list[dict]:
        """
        Detect character bounding boxes on a CAPTCHA-like image, plot boxes on the image
        and annotate each box with its size (width x height) and pixel area.

        Args:
            img: BGR image (numpy.ndarray) as read by cv2.imread.
            min_area: minimum component pixel area to keep.
            connectivity: 4 or 8 for connected components.
            s_thresh, v_thresh: thresholds to detect white background (same logic as other functions).
            display: if True show matplotlib plot, otherwise only return boxes.

        Returns:
            boxes: list of tuples (x, y, w, h, area) for each detected box (left, top, width, height, area).
        """
        # Convert to HSV and compute foreground mask (non-white pixels)
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        s_channel = hsv[:, :, 1]
        v_channel = hsv[:, :, 2]

        bg_mask = cv2.bitwise_and(
            cv2.compare(s_channel, s_thresh, cv2.CMP_LT),
            cv2.compare(v_channel, v_thresh, cv2.CMP_GT)
        )
        fg_mask = cv2.bitwise_not(bg_mask)  # 0 background, 255 foreground

        # Ensure binary 0/255
        _, fg_bin = cv2.threshold(fg_mask, 0, 255, cv2.THRESH_BINARY)

        # Connected components to get bounding boxes and areas
        num_components, comp_labels, stats, centroids = \
            cv2.connectedComponentsWithStats(fg_bin, connectivity, cv2.CV_32S)

        boxes = []
        for i in range(1, num_components):  # skip background label 0
            area = int(stats[i, cv2.CC_STAT_AREA])
            if area < min_area:
                continue
            x = int(stats[i, cv2.CC_STAT_LEFT])
            y = int(stats[i, cv2.CC_STAT_TOP])
            w = int(stats[i, cv2.CC_STAT_WIDTH])
            h = int(stats[i, cv2.CC_STAT_HEIGHT])
            boxes.append({'x': x, 'y': y, 'w': w, 'h': h, 'area': area})

        # Optionally display on matplotlib
        if display:
            # Convert BGR -> RGB for plotting
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            _, ax = plt.subplots(figsize=(10, 4))
            ax.imshow(img_rgb)
            ax.set_axis_off()

            for box in boxes:
                x, y, w, h, area = box['x'], box['y'], box['w'], box['h'], box['area']
                rect = Rectangle((x, y), w, h,
                                linewidth=1.5, edgecolor='r', facecolor='none')
                ax.add_patch(rect)
                # Annotate with size and area. Use a contrasting bbox for readability.
                label = f"{w}x{h}\n{area}px"
                ax.text(x, y - 2, label,
                        color='white', fontsize=8,
                        verticalalignment='bottom',
                        bbox=dict(facecolor='black', alpha=0.6, pad=1))
            plt.tight_layout()
            plt.show()

        return boxes

    def crop_and_resize_characters(
            self,
            img: np.ndarray,
            min_area: int = 20,
            padding: int = 5,
            dilate_kernel_size: int = 5,
            dilate_iters: int = 2,
            s_thresh: int = 5,
            v_thresh: int = 250,
            char_image_size: tuple[int, int] = (CHAR_IMAGE_WIDTH, CHAR_IMAGE_HEIGHT)
    ) -> tuple[list[np.ndarray], list[tuple[int,int,int,int,int]]]:
        """
        Detect loose character boundaries and return cropped character images.

        Returns:
        (crops, boxes)
        - crops: list of BGR cropped images (numpy.ndarray)
        - boxes: list of tuples (x, y, w, h, area)
        """
        h_img, w_img = img.shape[:2]

        # HSV foreground (non-white) mask (same logic used elsewhere)
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        s_channel = hsv[:, :, 1]
        v_channel = hsv[:, :, 2]
        bg_mask = cv2.bitwise_and(
            cv2.compare(s_channel, s_thresh, cv2.CMP_LT),
            cv2.compare(v_channel, v_thresh, cv2.CMP_GT)
        )
        fg_mask = cv2.bitwise_not(bg_mask)
        _, fg_bin = cv2.threshold(fg_mask, 0, 255, cv2.THRESH_BINARY)

        # Dilate to produce loose, merged regions
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilate_kernel_size, dilate_kernel_size))
        fg_loose = cv2.dilate(fg_bin, kernel, iterations=dilate_iters)

        # Find external contours
        contours, _ = cv2.findContours(fg_loose, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        boxes = []
        crops = []
        for cnt in contours:
            area = int(cv2.contourArea(cnt))
            if area < min_area:
                continue
            x, y, w, h = cv2.boundingRect(cnt)

            # expand by padding and clamp to image bounds
            x0 = max(0, x - padding)
            y0 = max(0, y - padding)
            x1 = min(w_img, x + w + padding)
            y1 = min(h_img, y + h + padding)

            boxes.append((x0, y0, x1 - x0, y1 - y0, area))
            crop = img[y0:y1, x0:x1].copy()
            crops.append(crop)

        # sort left-to-right by x coordinate
        if boxes:
            order = sorted(range(len(boxes)), key=lambda i: boxes[i][0])
            boxes = [boxes[i] for i in order]
            crops = [cv2.resize(crops[i], char_image_size) for i in order]

        # Convert crops to grayscale
        crops = [cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY) for crop in crops]
        return crops, boxes

    def segment_characters(self, img: np.ndarray) -> tuple[int, list[np.ndarray]]:
        """
        Segments characters from a CAPTCHA image using color clustering.

        Args:
            img: Input RGB (or BGR) image as NumPy array.
        Returns:
            (character_count, list_of_character_images).
        """
        processed_img = self.remove_lines(img)
        _, char_segmented_by_color = self.segment_by_color(processed_img)
        detected_infos = [self.detect_chars_and_sizes(char_img, display=False) for char_img in char_segmented_by_color]
        char_segmented = []
        avg_size = np.median([np.sum([box['area'] for box in info]) for info in detected_infos if len(info) > 0])

        for detected_info, char_img in zip(detected_infos, char_segmented_by_color):
            if len(detected_info) == 0:
                continue

            if len(detected_info) > 1 or detected_info[0]['area'] > avg_size * 1.2:
                _, char_segmented_by_value = self.segment_by_value(char_img)
                char_segmented.extend(char_segmented_by_value)
            else:
                char_segmented.append(char_img)

        char_segmented = [cropped_char_img for char_img in char_segmented for cropped_char_img in self.crop_and_resize_characters(char_img)[0]]
        return len(char_segmented), char_segmented

In [ ]:
# =====================================================================
# 1. DATA PROCESSING & SEGMENTATION
# =====================================================================
class CaptchaDataset(Dataset):
    """Dataset for CAPTCHA images with labels"""

    def __init__(self, segmented_images: list[np.ndarray], labels: list[str],
                char_to_idx: dict, idx_to_char: dict,
                segmenter: CaptchaSegmenter,
                char_transform = None,
                max_length=15):
        """
        Args:
            image_paths: List of paths to CAPTCHA images
            labels: List of text labels for CAPTCHAs
            char_vocab: String containing all possible characters (e.g., "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ")
            segmenter: CaptchaSegmenter instance
            char_transform: Transforms to apply to individual character images
        """
        self.segmented_images = segmented_images
        self.labels = labels
        self.char_to_idx = char_to_idx
        self.idx_to_char = idx_to_char
        self.segmenter = segmenter
        self.char_transform = char_transform
        self.max_length = max_length

    def __len__(self):
        return len(self.segmented_images)

    def __getitem__(self, idx):
        # Load image
        segmented_image = self.segmented_images[idx]
        chars_count = len(segmented_image)

        # Transform character
        char_tensors = []

        if self.char_transform:
            for char_img in segmented_image:
                char_img_pil = Image.fromarray(char_img)
                char_tensors.append(self.char_transform(char_img_pil))
        else:
            for char_img in segmented_image:
                char_img_pil = Image.fromarray(char_img)
                char_tensors.append(transforms.ToTensor()(char_img_pil))

        # Pad or truncate to max_length
        if chars_count < self.max_length:
            # Pad with zeros
            padding = [torch.zeros_like(char_tensors[0]) for _ in range(self.max_length - chars_count)]
            char_tensors.extend(padding)
        else:
            char_tensors = char_tensors[:self.max_length]

        # Stack into (max_length, C, H, W)
        char_sequence = torch.stack(char_tensors)

        # Convert label to indices
        label = self.labels[idx]
        label_indices = [self.char_to_idx[c] for c in label if c in self.char_to_idx]

        # Pad label
        if len(label_indices) < self.max_length:
            label_indices.extend([-1] * (self.max_length - len(label_indices)))
        else:
            label_indices = label_indices[:self.max_length]

        return char_sequence, torch.tensor(label_indices), chars_count

In [ ]:
class CharacterCNN(nn.Module):
    """Custom CNN for character recognition - corrected for 1-channel input"""

    def __init__(self, feature_output_dim: int = 256):
        super(CharacterCNN, self).__init__()

        self.features = nn.Sequential(
            # Conv Block 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),

            # Conv Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),

            # Conv Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),
        )

        # Adaptive pooling to handle variable input sizes
        # This will take the output of your conv blocks (which will be 128 x 8 x 6)
        # and resize it to (128 x 4 x 4), so the classifier input is correct.
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(128 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            # The last layer now outputs a feature vector, not num_classes
            nn.Linear(512, feature_output_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.adaptive_pool(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

In [ ]:
# =====================================================================
# 2. MODEL ARCHITECTURE
# =====================================================================

class CaptchaCRNN(nn.Module):
    """Combined CNN-RNN model for CAPTCHA recognition"""

    def __init__(self, num_classes: int, cnn_output_dim: int = 256,
                 hidden_size=256, num_layers=2, freeze_cnn=False):
        super(CaptchaCRNN, self).__init__()

        # Character-level CNN
        # Instantiate the CharacterCNNModel directly
        self.char_cnn = CharacterCNN(feature_output_dim=cnn_output_dim)

        if freeze_cnn:
            for param in self.char_cnn.parameters():
                param.requires_grad = False

        # RNN to process sequence of character features
        self.rnn = nn.LSTM(
            input_size=cnn_output_dim, # Match the CNN output
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.3 if num_layers > 1 else 0
        )

        self.fc = nn.Linear(hidden_size * 2, num_classes)  # *2 for bidirectional

    def forward(self, x, lengths=None):
        batch_size, seq_length, C, H, W = x.size()
        x = x.view(batch_size * seq_length, C, H, W)
        char_features = self.char_cnn(x)
        char_features = char_features.view(batch_size, seq_length, -1)

        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                char_features, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            rnn_out, _ = self.rnn(packed)
            rnn_out, _ = nn.utils.rnn.pad_packed_sequence(
                rnn_out, batch_first=True, total_length=char_features.size(1)
            )
        else:
            rnn_out, _ = self.rnn(char_features)

        output = self.fc(rnn_out)
        return output

In [ ]:
# =====================================================================
# 3. TRAINING
# =====================================================================
class CaptchaTrainer:
    """Trainer for CAPTCHA recognition model"""

    def __init__(self, model, device, CHAR_TO_IDX, IDX_TO_CHAR):
        self.model = model.to(device)
        self.device = device
        self.char_to_idx = CHAR_TO_IDX
        self.idx_to_char = IDX_TO_CHAR
        # ignore_index=-1 is correct for padded labels
        self.criterion = nn.CrossEntropyLoss(ignore_index=-1)

    def train_epoch(self, dataloader, optimizer, scheduler=None):
        self.model.train()
        total_loss = 0
        correct_chars = 0
        total_chars = 0

        for batch_idx, (images, labels, lengths) in enumerate(dataloader):
            images = images.to(self.device)
            labels = labels.to(self.device)
            lengths = lengths.to(self.device) # lengths are used by the model

            optimizer.zero_grad()

            # Forward pass
            # outputs shape is (batch, seq, num_classes)
            outputs = self.model(images, lengths)

            # --- SIMPLIFIED LOSS CALCULATION ---
            # Reshape for loss calculation.
            # We are guaranteed outputs.size(1) == labels.size(1)
            # thanks to our model's forward pass logic.
            outputs_flat = outputs.view(-1, outputs.size(-1))
            labels_flat = labels.view(-1)

            # Calculate loss (criterion ignores labels == -1)
            loss = self.criterion(outputs_flat, labels_flat)

            # Backward pass
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=5.0)
            optimizer.step()


            # --- SIMPLIFIED ACCURACY CALCULATION ---
            _, predicted = outputs_flat.max(1)

            # Create mask to ignore padding
            mask = labels_flat != -1

            correct_chars += (predicted[mask] == labels_flat[mask]).sum().item()
            total_chars += mask.sum().item() # Only count non-padded chars

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        accuracy = correct_chars / total_chars if total_chars > 0 else 0

        return avg_loss, accuracy

    def validate(self, dataloader):
        self.model.eval()
        total_loss = 0
        correct_chars = 0
        total_chars = 0
        correct_captchas = 0
        total_captchas = 0

        with torch.no_grad():
            for images, labels, lengths in dataloader:
                images = images.to(self.device)
                labels = labels.to(self.device)
                lengths = lengths.to(self.device)

                # Forward pass
                outputs = self.model(images, lengths)

                # --- SIMPLIFIED LOSS & CHAR ACCURACY ---
                outputs_flat = outputs.view(-1, outputs.size(-1))
                labels_flat = labels.view(-1)

                # Calculate loss
                loss = self.criterion(outputs_flat, labels_flat)
                total_loss += loss.item()

                # Calculate character-level accuracy
                _, predicted = outputs_flat.max(1)
                mask = labels_flat != -1
                correct_chars += (predicted[mask] == labels_flat[mask]).sum().item()
                total_chars += mask.sum().item()

                # --- ROBUST CAPTCHA-LEVEL ACCURACY (Your code is good!) ---
                pred_labels = outputs.argmax(dim=2)  # (batch, seq)
                for i in range(images.size(0)): # Use images.size(0) for batch_size
                    # Find the true length of the label
                    original_label_length = (labels[i] != -1).sum().item()

                    if original_label_length > 0 and torch.all(
                        pred_labels[i, :original_label_length] == labels[i, :original_label_length]
                    ):
                         correct_captchas += 1
                    total_captchas += 1

        avg_loss = total_loss / len(dataloader)
        char_accuracy = correct_chars / total_chars if total_chars > 0 else 0
        captcha_accuracy = correct_captchas / total_captchas if total_captchas > 0 else 0

        return avg_loss, char_accuracy, captcha_accuracy


    def predict(self, image_path, segmenter, transform):
        """Predict CAPTCHA text from image"""
        self.model.eval()

        # Load and segment image
        image = cv2.imread(image_path)
        if image is None:
            print(f"Error: Could not load image from {image_path}")
            return ""

        image_np = np.array(image)
        # Assuming segment_characters returns the segmented images
        _, char_images = segmenter.segment_characters(image_np)

        # Transform characters
        char_tensors = []
        for char_img in char_images:
            # --- BUG FIX ---
            # REMOVED the cv2.cvtColor(..., cv2.COLOR_GRAY2RGB)
            # Our model 'CharacterCNNModel' expects 1-channel input.
            # The 'transform' (which should include ToTensor())
            # will handle converting the (H, W) numpy array
            # to a (1, H, W) tensor.
            char_tensor = transform(char_img).to(self.device)
            char_tensors.append(char_tensor)

        if len(char_tensors) == 0:
            return ""

        # Stack and add batch dimension
        # Shape becomes (1, seq, C, H, W)
        char_sequence = torch.stack(char_tensors).unsqueeze(0).to(self.device)
        # Create the lengths tensor for the model
        num_chars = torch.tensor([len(char_tensors)], dtype=torch.long).to(self.device)

        # Predict
        with torch.no_grad():
            # Pass both the images and their lengths
            outputs = self.model(char_sequence, num_chars)
            # Get predictions (batch_size=1, seq_len, num_classes)
            predictions = outputs.argmax(dim=2).squeeze(0) # Squeeze batch dim

        # Convert to text
        # Only convert up to the number of characters we found
        predicted_text = ''.join(
            [self.idx_to_char[idx.item()] for idx in predictions]
        )

        return predicted_text

In [ ]:
def loads_imgs_and_labels_from_dir(data_dir: str, need_broken_images: bool = False) -> tuple[list[np.ndarray], list[str]]:
    """Loads image file paths and labels from a directory."""
    broken_images_path = []
    segmented_images = []
    labels = []

    for (root, _, files) in os.walk(data_dir, topdown=True):
        for file in tqdm(files, desc=f"Loading '{data_dir}' images"):
            if not file.endswith('.png'):
                continue
            if file.split(".png")[0].endswith('(1)'): # Remove duplicate data
                continue
            if file.endswith('badwrong.png'): # Remove known bad data
                continue
            if file.endswith('badcrop.png'):
                continue
            if file.endswith('badmiss.png'):
                continue

            img_path = os.path.join(root, file)
            label = file.split(".")[0].split("-")[0]

            try:
                img = cv2.imread(img_path)
                img_np = np.array(img)
                _, segmented_image = CaptchaSegmenter().segment_characters(img_np)
                segmented_images.append(segmented_image)
                labels.append(label)
            except Exception as e:
                broken_images_path.append(img_path)
                continue

    if need_broken_images:
        with open("broken_images.txt", "w") as f:
            for path in broken_images_path:
                f.write(f"{path}\n")

    return segmented_images, labels


In [ ]:
BATCH_SIZE = 32
NUM_EPOCHS = 70
LEARNING_RATE = 0.001
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MAX_LENGTH = 12

print("Device:", DEVICE)

Device: cuda


In [ ]:
# Initialize segmenter
segmenter = CaptchaSegmenter()

all_img_paths, all_labels = loads_imgs_and_labels_from_dir('./cleaned_train', True)
train_images, val_images, train_labels, val_labels = \
    train_test_split(all_img_paths, all_labels, train_size=0.8, random_state=42)

Loading './cleaned_train' images: 100%|██████████| 8001/8001 [05:51<00:00, 22.74it/s]


In [ ]:
char_transform = transforms.Compose([
    transforms.Resize((CHAR_IMAGE_HEIGHT, CHAR_IMAGE_WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])  # Grayscale normalization
])

train_dataset = CaptchaDataset(
    train_images,
    train_labels,
    CHAR_TO_IDX,
    IDX_TO_CHAR,
    segmenter,
    char_transform,
    MAX_LENGTH
)

val_dataset = CaptchaDataset(
    val_images,
    val_labels,
    CHAR_TO_IDX,
    IDX_TO_CHAR,
    segmenter,
    char_transform,
    MAX_LENGTH
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                            shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                        shuffle=False)

In [ ]:
model = CaptchaCRNN(
    num_classes=NUM_CLASSES,
    hidden_size=256,
    num_layers=2,
    freeze_cnn=False  # Set to True to only train RNN initially
)

# Initialize trainer
trainer = CaptchaTrainer(model, DEVICE, CHAR_TO_IDX, IDX_TO_CHAR)

# Optimizer and scheduler
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

# Training loop
best_val_accuracy = 0
train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = trainer.train_epoch(train_loader, optimizer)
    val_loss, val_char_acc, val_captcha_acc = trainer.validate(val_loader)

    scheduler.step(val_loss)

    # Record losses and accuracies
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_captcha_acc)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"  Val Loss: {val_loss:.4f}, Val Char Acc: {val_char_acc:.4f}, "
            f"Val CAPTCHA Acc: {val_captcha_acc:.4f}")

    # Save best model
    if val_captcha_acc > best_val_accuracy:
        best_val_accuracy = val_captcha_acc
        torch.save(model.state_dict(), 'best_captcha_model.pth')
        print(f"  Saved best model with CAPTCHA accuracy: {val_captcha_acc:.4f}")

print(f"\nTraining completed. Best validation CAPTCHA accuracy: {best_val_accuracy:.4f}")

Epoch 1/70
  Train Loss: 3.4722, Train Acc: 0.0446
  Val Loss: 3.5305, Val Char Acc: 0.0692, Val CAPTCHA Acc: 0.0000
Epoch 2/70
  Train Loss: 2.6299, Train Acc: 0.2194
  Val Loss: 1.9367, Val Char Acc: 0.4249, Val CAPTCHA Acc: 0.0262
  Saved best model with CAPTCHA accuracy: 0.0262
Epoch 3/70
  Train Loss: 1.8202, Train Acc: 0.4604
  Val Loss: 1.4203, Val Char Acc: 0.5944, Val CAPTCHA Acc: 0.1234
  Saved best model with CAPTCHA accuracy: 0.1234
Epoch 4/70
  Train Loss: 1.4754, Train Acc: 0.5664
  Val Loss: 1.2660, Val Char Acc: 0.6500, Val CAPTCHA Acc: 0.2001
  Saved best model with CAPTCHA accuracy: 0.2001
Epoch 5/70
  Train Loss: 1.2980, Train Acc: 0.6214
  Val Loss: 1.0966, Val Char Acc: 0.7008, Val CAPTCHA Acc: 0.2519
  Saved best model with CAPTCHA accuracy: 0.2519
Epoch 6/70
  Train Loss: 1.1574, Train Acc: 0.6646
  Val Loss: 1.0445, Val Char Acc: 0.7116, Val CAPTCHA Acc: 0.2871
  Saved best model with CAPTCHA accuracy: 0.2871
Epoch 7/70
  Train Loss: 1.0550, Train Acc: 0.6953
  

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss over Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label='Train CAPTCHA Acc')
plt.plot(val_accuracies, label='Val CAPTCHA Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('CAPTCHA Accuracy over Epochs')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
test_imgs, test_labels = loads_imgs_and_labels_from_dir('./cleaned_test')
test_dataset = CaptchaDataset(
    test_imgs,
    test_labels,
    CHAR_TO_IDX,
    IDX_TO_CHAR,
    segmenter, MAX_LENGTH
)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
test_loss, test_char_acc, test_captcha_acc = trainer.validate(test_loader)

print(f"Test Loss: {test_loss:.4f}, Test Char Acc: {test_char_acc:.4f}, Test CAPTCHA Acc: {test_captcha_acc:.4f}")

Loading './cleaned_test' images: 100%|██████████| 2001/2001 [01:25<00:00, 23.41it/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Test Loss: 0.8664, Test Char Acc: 0.7876, Test CAPTCHA Acc: 0.4050
